# DRfold — Verified Google Colab Workflow

**Project:** RNA 3D Structure Prediction for Template-Sparse RNA Sequences  
**Model:** DRfold (original)  
**Official repo:** `leeyang/DRfold`  
**Paper:** Nature Communications (2023)  
**DOI:** `10.1038/s41467-023-41303-9`

Verified bundled-test run:
- commit `9586990c79e4ca488e5f967fcc8bd5b06cd60273`
- Python `3.9.19`
- NumPy `1.23.5`
- SciPy `1.13.1`
- PyTorch `1.13.1+cu117`
- OpenMM `7.7`
- ViennaRNA `2.4.18`
- PETfold `2.0`
- Tesla T4
- runtime `5669.692 s`
- peak observed GPU memory `1395 MiB`
- final output `DPR.pdb`
- bundled target length `77 nt`

Primary DRfold benchmark input is **RNA FASTA only**. PETfold + ViennaRNA features are generated internally.

## 1. Host / GPU check

In [ ]:
import os, sys, time, threading, subprocess, platform, re, csv, shutil
from pathlib import Path

print("Host Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

## 2. Clone official DRfold repository

In [ ]:
%cd /content
REPO = Path("/content/DRfold")
if not REPO.exists():
    !git clone https://github.com/leeyang/DRfold.git

%cd /content/DRfold
commit = subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip()
print("DRfold commit:", commit)

## 3. Install Micromamba

In [ ]:
%cd /content
MAMBA = Path("/content/bin/micromamba")
if not MAMBA.exists():
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj
!/content/bin/micromamba --version

## 4. Create base environment

In [ ]:
MAMBA = "/content/bin/micromamba"
ROOT = "/content/micromamba_root"
ENV = "drfold"

subprocess.run([MAMBA,"-r",ROOT,"env","remove","-n",ENV,"-y"],
               check=False,capture_output=True,text=True)

p = subprocess.run([
    MAMBA,"-r",ROOT,"create","-y","-n",ENV,
    "-c","conda-forge",
    "python=3.9","numpy=1.23.5","scipy","openmm=7.7.0","pip","pkg-config"
], capture_output=True, text=True)

print(p.stdout); print(p.stderr)
if p.returncode != 0:
    raise RuntimeError("Base environment creation failed.")

## 5. Install ViennaRNA 2.4.18

In [ ]:
p = subprocess.run([
    MAMBA,"-r",ROOT,"install","-y","-n",ENV,
    "-c","bioconda","-c","conda-forge","viennarna=2.4.18"
], capture_output=True, text=True)

print(p.stdout); print(p.stderr)
if p.returncode != 0:
    raise RuntimeError("ViennaRNA installation failed.")

## 6. Install PyTorch 1.13.1 + CUDA 11.7

In [ ]:
p = subprocess.run([
    MAMBA,"-r",ROOT,"run","-n",ENV,
    "pip","install","torch==1.13.1+cu117",
    "--extra-index-url","https://download.pytorch.org/whl/cu117"
], capture_output=True, text=True)

print(p.stdout); print(p.stderr)
if p.returncode != 0:
    raise RuntimeError("PyTorch installation failed.")

## 7. Verify environment

In [ ]:
verify_code = r'''
import sys, shutil, numpy, scipy, torch, openmm
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("OpenMM:", openmm.__version__)
print("RNAfold:", shutil.which("RNAfold"))
'''
v = subprocess.run([MAMBA,"-r",ROOT,"run","-n",ENV,"python","-c",verify_code],
                   capture_output=True,text=True)
print(v.stdout); print(v.stderr)
if v.returncode != 0:
    raise RuntimeError("Environment verification failed.")

VIENNA_PREFIX = Path("/content/micromamba_root/envs/drfold")
RNAFOLD_EXE = VIENNA_PREFIX / "bin" / "RNAfold"

## 8. Install build tools

In [ ]:
!apt-get update -qq
!apt-get install -y -qq build-essential clang wget unzip tar

## 9. Download and build PETfold 2.0

In [ ]:
%cd /content

PET_ARCHIVE = Path("/content/PETfold2.0.tar.gz")
PET_EXTRACT = Path("/content/PETfold2_source")

if not PET_ARCHIVE.exists():
    !wget -O /content/PETfold2.0.tar.gz https://rth.dk/resources/petfold/download/PETfold2.0.tar.gz

if PET_EXTRACT.exists():
    shutil.rmtree(PET_EXTRACT)
PET_EXTRACT.mkdir(parents=True)

subprocess.run(["tar","-xzf",str(PET_ARCHIVE),"-C",str(PET_EXTRACT)], check=True)

make_dirs = [p for p in PET_EXTRACT.rglob("*")
             if p.is_dir() and (p/"Makefile").exists()]
if not make_dirs:
    raise FileNotFoundError("PETfold Makefile not found.")

PET_SRC = make_dirs[0]
MAKEFILE = PET_SRC / "Makefile"
VIENNA_INCLUDE = VIENNA_PREFIX / "include" / "ViennaRNA"
VIENNA_LIB = VIENNA_PREFIX / "lib"

txt = MAKEFILE.read_text()
txt = re.sub(r"^HPATH\s*=.*$", f"HPATH = {VIENNA_INCLUDE}", txt, flags=re.MULTILINE)
txt = re.sub(r"^LPATH\s*=.*$", f"LPATH = -L{VIENNA_LIB} -lRNA -lstdc++", txt, flags=re.MULTILINE)

fixed = []
for line in txt.splitlines():
    if not line.lstrip().startswith("#") and "gcc " in line and "-fcommon" not in line:
        line = re.sub(r"\bgcc\s+", "gcc -fcommon ", line)
    fixed.append(line)
MAKEFILE.write_text("\n".join(fixed)+"\n")

build_env = os.environ.copy()
build_env["PATH"] = f"{VIENNA_PREFIX/'bin'}:" + build_env["PATH"]
build_env["CPATH"] = f"{VIENNA_INCLUDE}:{VIENNA_PREFIX/'include'}:" + build_env.get("CPATH","")
build_env["LIBRARY_PATH"] = f"{VIENNA_LIB}:" + build_env.get("LIBRARY_PATH","")
build_env["LD_LIBRARY_PATH"] = f"{VIENNA_LIB}:" + build_env.get("LD_LIBRARY_PATH","")

subprocess.run(["make","clean"], cwd=str(PET_SRC), env=build_env,
               capture_output=True,text=True)

b = subprocess.run(["make","../bin/PETfold"], cwd=str(PET_SRC), env=build_env,
                   capture_output=True,text=True)

print(b.stdout[-12000:]); print(b.stderr[-12000:])
PETFOLD_REAL = PET_SRC.parent / "bin" / "PETfold"

if b.returncode != 0 or not PETFOLD_REAL.exists():
    raise RuntimeError("PETfold compilation failed.")
PETFOLD_REAL.chmod(PETFOLD_REAL.stat().st_mode | 0o111)
print("PETfold:", PETFOLD_REAL)

## 10. Create PETfold runtime wrapper

In [ ]:
scfg_matches = list(PET_SRC.parent.rglob("scfg.rate"))
if not scfg_matches:
    raise FileNotFoundError("scfg.rate not found.")

SCFG_RATE = scfg_matches[0]
PETFOLD_DATA_DIR = SCFG_RATE.parent
WRAPPER = PET_SRC.parent / "PETfold_wrapper.sh"

WRAPPER.write_text(
    f'''#!/bin/bash
cd "{PETFOLD_DATA_DIR}"
exec "{PETFOLD_REAL}" "$@"
'''
)
WRAPPER.chmod(0o755)
PETFOLD = WRAPPER
print("PETfold wrapper:", PETFOLD)

## 11. Verify Arena

In [ ]:
%cd /content/DRfold
ARENA = Path("/content/DRfold/bin/Arena")
ARENA.chmod(ARENA.stat().st_mode | 0o111)
p = subprocess.run([str(ARENA)],capture_output=True,text=True)
print("Return code:",p.returncode)
print(p.stderr[:3000])
if not ARENA.exists():
    raise FileNotFoundError("Arena missing.")

## 12. Download official DRfold model weights

In [ ]:
%cd /content

MODEL_ZIP = Path("/content/DRfold_models.zip")
MODEL_TMP = Path("/content/DRfold_models_unpack")

if not MODEL_ZIP.exists():
    !wget -O /content/DRfold_models.zip https://zhanggroup.org/DRfold/DRfold_models.zip

if MODEL_TMP.exists():
    shutil.rmtree(MODEL_TMP)
MODEL_TMP.mkdir(parents=True)

subprocess.run(["unzip","-q","-o",str(MODEL_ZIP),"-d",str(MODEL_TMP)], check=True)

MODEL_ROOT = None
for p in [MODEL_TMP] + [x for x in MODEL_TMP.rglob("*") if x.is_dir()]:
    if (p/"DeepE2EPotential").exists() and (p/"DeepGeoPotential").exists():
        MODEL_ROOT = p
        break

if MODEL_ROOT is None:
    raise FileNotFoundError("Model directories not found.")

shutil.copytree(MODEL_ROOT, Path("/content/DRfold"), dirs_exist_ok=True)

required = [
    Path("/content/DRfold/DeepE2EPotential/model"),
    Path("/content/DRfold/DeepGeoPotential/config1/model"),
    Path("/content/DRfold/DeepGeoPotential/config2/model"),
    Path("/content/DRfold/DeepGeoPotential/config3/model"),
]
if not all(p.exists() for p in required):
    raise FileNotFoundError("Incomplete DRfold model files.")
print("Model files ready.")

## 13. Patch DRfold paths

In [ ]:
%cd /content/DRfold

FEATURE_PY = Path("/content/DRfold/scripts/Feature.py")
DRFOLD_SH = Path("/content/DRfold/DRfold.sh")
PYTHON_EXE = VIENNA_PREFIX / "bin" / "python"

t = FEATURE_PY.read_text()
t = re.sub(r"petfoldbin\s*=.*", f"petfoldbin = r'{PETFOLD}'", t)
t = re.sub(r"ViennaRNAbin\s*=.*", f"ViennaRNAbin = r'{RNAFOLD_EXE}'", t)
FEATURE_PY.write_text(t)

s = DRFOLD_SH.read_text()
s = re.sub(r'export PETFOLDBIN=.*', f'export PETFOLDBIN="{PETFOLD_REAL.parent}"', s)
s = re.sub(r'^PYTHON=.*$', f'PYTHON="{PYTHON_EXE}"', s, flags=re.MULTILINE)
DRFOLD_SH.write_text(s)
DRFOLD_SH.chmod(DRFOLD_SH.stat().st_mode | 0o111)
print("Paths patched.")

## 14. Feature preflight

In [ ]:
TEST_FASTA = Path("/content/DRfold/test/seq.fasta")
PREFLIGHT_DIR = Path("/content/DRfold/test/preflight")
if PREFLIGHT_DIR.exists():
    shutil.rmtree(PREFLIGHT_DIR)
PREFLIGHT_DIR.mkdir(parents=True)

PREFIX = PREFLIGHT_DIR / "seq_ss"
NORMALIZED_FASTA = PREFLIGHT_DIR / "seq.fasta"

preflight_env = os.environ.copy()
preflight_env["PATH"] = f"{VIENNA_PREFIX/'bin'}:{PETFOLD_REAL.parent}:" + preflight_env["PATH"]
preflight_env["LD_LIBRARY_PATH"] = f"{VIENNA_PREFIX/'lib'}:" + preflight_env.get("LD_LIBRARY_PATH","")

p = subprocess.run([
    str(PYTHON_EXE), str(FEATURE_PY), str(TEST_FASTA),
    str(PREFIX), str(NORMALIZED_FASTA)
], cwd="/content/DRfold", env=preflight_env, capture_output=True, text=True)

print("RETURN CODE:", p.returncode)
print(p.stdout); print(p.stderr)

FEATURE_NPY = Path(str(PREFIX)+".npy")
print("Feature tensor exists:", FEATURE_NPY.exists())

if p.returncode != 0 or not FEATURE_NPY.exists():
    raise RuntimeError("Feature preflight failed.")

## 15. Run bundled DRfold inference

In [ ]:
%cd /content/DRfold

VERIFY_OUT = Path("/content/DRfold/test/DRfold_out_colab")
if VERIFY_OUT.exists():
    shutil.rmtree(VERIFY_OUT)

gpu_samples = []
stop_monitor = False

def monitor_gpu():
    global stop_monitor
    while not stop_monitor:
        try:
            x = subprocess.check_output(
                ["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"],
                text=True
            ).strip().splitlines()
            if x:
                gpu_samples.append(int(x[0]))
        except Exception:
            pass
        time.sleep(1)

mon = threading.Thread(target=monitor_gpu, daemon=True)
mon.start()

run_env = os.environ.copy()
run_env["PATH"] = f"{VIENNA_PREFIX/'bin'}:{PETFOLD_REAL.parent}:" + run_env["PATH"]
run_env["LD_LIBRARY_PATH"] = f"{VIENNA_PREFIX/'lib'}:" + run_env.get("LD_LIBRARY_PATH","")

start = time.perf_counter()
proc = subprocess.run(
    ["bash",str(DRFOLD_SH),str(TEST_FASTA),str(VERIFY_OUT)],
    cwd="/content/DRfold", env=run_env, capture_output=True, text=True
)
elapsed = time.perf_counter() - start

stop_monitor = True
mon.join(timeout=3)
peak_gpu_mb = max(gpu_samples) if gpu_samples else None

print(proc.stdout); print(proc.stderr)
print("Exit code:", proc.returncode)
print(f"Runtime: {elapsed:.2f} s")
print("Peak observed GPU memory:", peak_gpu_mb, "MiB")

if proc.returncode != 0:
    raise RuntimeError("DRfold inference failed.")

## 16. Verify output and save metadata

In [ ]:
FINAL_PDB = VERIFY_OUT / "DPR.pdb"
print("Final PDB:", FINAL_PDB)
print("Exists:", FINAL_PDB.exists())

if not FINAL_PDB.exists():
    raise FileNotFoundError("DPR.pdb not generated.")

record = {
    "model":"DRfold",
    "model_commit":commit,
    "target":"bundled_test_seq",
    "input_fasta":"test/seq.fasta",
    "output":str(FINAL_PDB),
    "runtime_seconds":round(elapsed,3),
    "peak_gpu_memory_mib_observed":peak_gpu_mb,
    "exit_code":proc.returncode
}
RUN_LOG_CSV = Path("/content/drfold_run_log.csv")
with RUN_LOG_CSV.open("w",newline="") as f:
    w = csv.DictWriter(f, fieldnames=record.keys())
    w.writeheader(); w.writerow(record)

print(record)

## 17. Optional — save to Google Drive

In [ ]:
# OPTIONAL
# from google.colab import drive
# drive.mount("/content/drive")
#
# DEST = Path("/content/drive/MyDrive/RNA3D_Benchmark/DRfold")
# DEST.mkdir(parents=True, exist_ok=True)
# shutil.copytree(VERIFY_OUT, DEST/"bundled_test", dirs_exist_ok=True)
# shutil.copy2(RUN_LOG_CSV, DEST/RUN_LOG_CSV.name)

# Custom RNA target

## 18. Enter custom sequence

In [ ]:
JOB_NAME = "my_rna"
RNA_SEQUENCE = """
GGGAAACCCUUUGGGAAACCC
""".replace("\n","").replace(" ","").upper()

if not re.fullmatch(r"[ACGU]+", RNA_SEQUENCE):
    raise ValueError("Sequence must contain only A/C/G/U.")

CUSTOM_INPUT = Path("/content/DRfold/custom_input")
CUSTOM_INPUT.mkdir(parents=True, exist_ok=True)
CUSTOM_FASTA = CUSTOM_INPUT / f"{JOB_NAME}.fasta"
CUSTOM_FASTA.write_text(f">{JOB_NAME}\n{RNA_SEQUENCE}\n")

print("Target:", JOB_NAME)
print("Length:", len(RNA_SEQUENCE))

## 19. Run custom inference

In [ ]:
CUSTOM_OUT = Path(f"/content/DRfold/custom_output/{JOB_NAME}")
if CUSTOM_OUT.exists():
    shutil.rmtree(CUSTOM_OUT)

custom_start = time.perf_counter()
custom_proc = subprocess.run(
    ["bash",str(DRFOLD_SH),str(CUSTOM_FASTA),str(CUSTOM_OUT)],
    cwd="/content/DRfold", env=run_env, capture_output=True, text=True
)
custom_elapsed = time.perf_counter() - custom_start

print(custom_proc.stdout); print(custom_proc.stderr)
print("Exit code:", custom_proc.returncode)
print(f"Runtime: {custom_elapsed:.2f} s")

CUSTOM_PDB = CUSTOM_OUT / "DPR.pdb"
if custom_proc.returncode != 0 or not CUSTOM_PDB.exists():
    raise RuntimeError("Custom DRfold inference failed.")

print("Custom PDB:", CUSTOM_PDB)

## Benchmark policy

Primary condition:

```text
RNA FASTA
→ PETfold + ViennaRNA SS features
→ geometry + end-to-end potentials
→ PotentialFold optimization
→ Arena all-atom reconstruction
→ OpenMM refinement
→ DPR.pdb
```

Use the same DRfold commit/model archive and record runtime, hardware, peak VRAM, and failures. Keep DRfold and DRfold2 as separate methods.